# Python. Работа с бинарными данными


## Мотивация

Текстовый файл удобен человеку, но миллион чисел в нём занимает много места и после чтения требует заново восстанавливать форму и типы. Бинарный формат хранит данные по заранее известным правилам: NumPy восстанавливает массив вместе с `shape` и `dtype`, HDF5 читает отдельные участки большого набора, а архив собирает несколько файлов для передачи.

Универсального формата нет. Сначала нужно понять, что именно сохраняется, кто будет читать файл и нужно ли загружать данные частями.


## 0. Подготовка

```bash
uv add numpy h5py
```


## 1. Pickle: удобно, но только для доверенных данных

`pickle` сохраняет состояние Python-объекта и инструкции, по которым Python восстановит его при `pickle.load`. Поэтому словарь со списками, массивами и экземплярами классов можно записать почти без подготовки.

Файл открывают в бинарном режиме: `"wb"` для записи и `"rb"` для чтения.

Главная опасность: загрузка pickle может вызвать указанные в файле функции. Злоумышленник способен подготовить файл так, что код выполнится уже во время `pickle.load`. Неизвестный pickle нельзя «просто посмотреть» — опасное действие начинается при загрузке. **Pickle может содержать не только данные, но и вызываемые инструкции восстановления, в том числе вредоносный вызов.**

Метод `__reduce__` может вернуть вызываемый объект и его аргументы. При загрузке pickle вызовет этот объект. В демонстрации используется безопасный `print`, чтобы выполнение было видно; злоумышленник может указать функцию с опасными действиями.

Код класса также обычно не замораживается внутри файла. Pickle хранит имя модуля, имя класса и состояние объекта. Если класс с тем же именем изменили, загруженный объект получит текущую реализацию методов. Поэтому pickle подходит только для доверенного внутреннего состояния в контролируемом окружении.


In [ ]:
from pathlib import Path
import pickle

WORK = Path("/tmp/course-binary")
WORK.mkdir(exist_ok=True)

class RunOnLoad:
    def __reduce__(self):
        return (
            print,
            ("ОПАСНОСТЬ: функция выполнилась внутри pickle.load()",),
        )

payload = {
    "title": "Продажи",
    "action": RunOnLoad(),
}

path = WORK / "dangerous.pkl"
with path.open("wb") as file:
    pickle.dump(payload, file)

print("До pickle.load")
with path.open("rb") as file:
    loaded = pickle.load(file)
print("После pickle.load")

print(loaded)


#### ❓ **Вопрос**

Между строками «До pickle.load» и «После pickle.load» появилось сообщение, хотя программа явно не вызывала `print` в этом месте. Откуда взялся вызов и чему стало равно поле `loaded["action"]`?

<details>
<summary>Ответ</summary>

`RunOnLoad.__reduce__` записал в pickle указание вызвать `print` с заданной строкой. `pickle.load` выполнил этот вызов при восстановлении объекта. `print` возвращает `None`, поэтому поле `loaded["action"]` стало равно `None`. Вместо `print` вредоносный файл может указать опасную функцию.

</details>


## 2. Создание массивов и типы

`ndarray` — массив элементов одного типа. Частые способы создания:

- `np.array(data)` — из списка или другой последовательности;
- `np.arange(start, stop, step)` — значения с заданным шагом;
- `np.linspace(start, stop, num)` — заданное число равномерных точек, включая границы;
- `np.zeros(shape)`, `np.ones(shape)` — массивы из нулей или единиц;
- `np.zeros_like(array)`, `np.ones_like(array)` — та же форма и обычно тот же тип, что у образца.

`np.random.seed(seed)` задаёт начальное состояние генератора псевдослучайных чисел, который используется в `np.random` функциях, если необходимо получать одинаковые последовательности, при разных запусках. Это используют для повторяемых примеров и тестов; криптографически безопасными числа от этого не становятся.

`np.concatenate(arrays, axis)` соединяет несколько массивов. У них должно быть одинаковое число осей, а размеры всех осей, кроме оси соединения, должны совпадать.

Например:

- формы `(2, 3)` и `(1, 3)` можно соединить по `axis=0`: второй размер совпадает, результат имеет форму `(3, 3)`;
- формы `(2, 3)` и `(2, 4)` можно соединить по `axis=1`: первый размер совпадает, результат имеет форму `(2, 7)`;
- формы `(2, 3)` и `(1, 3)` нельзя соединить по `axis=1`, потому что размеры другой оси — `2` и `1` — различаются.

Главные свойства массива: `shape`, `ndim`, `dtype`, `size`, `nbytes`. Тип задают при создании через `dtype=` или меняют через `array.astype(np.float32)`. `astype` возвращает преобразованный массив; исходный массив остаётся прежнего типа.


In [ ]:
import numpy as np

zeros = np.zeros((2, 3), dtype=np.float32)
ones = np.ones((1, 3), dtype=np.float32)
same_zeros = np.zeros_like(ones)
same_ones = np.ones_like(zeros)

grid = np.linspace(0, 1, num=5)
integers = np.arange(6).reshape(2, 3)
floats = integers.astype(np.float32)
combined = np.concatenate([zeros, ones], axis=0)

np.random.seed(7)
sample = np.random.normal(size=5)

print("grid:", grid)
print("integers:", integers.dtype, "floats:", floats.dtype)
print("combined shape:", combined.shape)
print("same shapes:", same_zeros.shape, same_ones.shape)
print("bytes:", combined.nbytes)
print("random sample:", sample)


#### ❓ **Вопрос**

Нужно соединить массивы форм `(2, 3)` и `(1, 3)` по `axis=0`, а затем делить значения пополам. Какой будет форма результата и зачем перед делением может понадобиться `astype(np.float32)`?

<details>
<summary>Ответ</summary>

Форма станет `(3, 3)`: складывается размер первой оси, остальные размеры совпадают. `astype(np.float32)` явно переводит целые данные в вещественный тип, если результат должен хранить дробные значения.

</details>


## 3. Размерности и оси

`shape` показывает размеры массива по каждой оси. У матрицы формы `(3, 4)`:

- ось `0` соответствует трём строкам;
- ось `1` соответствует четырём столбцам.

`reshape` меняет форму массива, не меняя количество элементов. Например, 12 последовательных значений можно представить как `(3, 4)`, `(2, 6)` или `(2, 2, 3)`. Вызов `reshape` не превращает бывшие столбцы в строки: он заново группирует элементы в указанную форму, сохраняя их последовательность.

Ось можно добавить:

- `vector[np.newaxis, :]` превращает вектор формы `(4,)` в одну строку формы `(1, 4)`;
- `vector[:, None]` превращает его в один столбец формы `(4, 1)`;
- `np.expand_dims(vector, axis=0)` явно добавляет ось и также даёт `(1, 4)`.

`transpose` переставляет оси. Для матрицы `matrix.T` или `matrix.transpose(1, 0)` меняет строки и столбцы местами: форма `(3, 4)` становится `(4, 3)`. Для массива с большим числом осей запись `np.transpose(array, (2, 0, 1))` означает: старая ось 2 станет первой, старая ось 0 — второй, старая ось 1 — третьей.

`np.moveaxis(array, source, destination)` переносит одну выбранную ось. Например, у изображений можно перенести ось цветовых каналов с последнего места на второе.

`flatten()` и `ravel()` превращают массив любой формы в одномерный массив формы `(size,)`: у результата остаётся одна последовательность элементов без строк и столбцов. `flatten()` всегда создаёт независимую копию. `ravel()` по возможности использует ту же память, поэтому изменение результата иногда меняет исходный массив.


In [ ]:
import numpy as np

values = np.arange(12)
matrix = values.reshape(3, 4)
another_shape = matrix.reshape(2, 6)

row = values[np.newaxis, :]
column = values[:, None]
same_row = np.expand_dims(values, axis=0)

images = np.zeros((8, 64, 64, 3), dtype=np.uint8)
channels_first = np.moveaxis(images, -1, 1)

transposed = matrix.T
one_dimension_copy = matrix.flatten()
one_dimension_maybe_shared = matrix.ravel()

print("values:", values.shape)
print("matrix:", matrix.shape, "reshaped:", another_shape.shape)
print("row:", row.shape, "column:", column.shape, "expand_dims:", same_row.shape)
print("images:", images.shape, "channels first:", channels_first.shape)
print("transpose:", transposed.shape)
print("one dimension:", one_dimension_copy.shape, one_dimension_maybe_shared.shape)


#### ❓ **Вопрос**

Массив содержит 12 элементов и имеет форму `(3, 4)`. Чем отличаются результаты `matrix.reshape(2, 6)` и `matrix.T`? Что произойдёт с формой после `matrix.flatten()`?

<details>
<summary>Ответ</summary>

`reshape(2, 6)` сохраняет последовательность элементов и группирует её в две строки по шесть значений. `matrix.T` меняет две оси местами: бывшие строки становятся столбцами, поэтому форма равна `(4, 3)`. `flatten()` создаёт одномерный массив формы `(12,)`.

</details>


## 4. Маски и частые вычисления

Арифметика и сравнения применяются сразу ко всему массиву. Выражение `data > 22` создаёт логическую маску той же формы, а запись `data[data > 22]` сразу возвращает все подходящие элементы.

Частые операции:

- `sum`, `mean`, `min`, `max`, `std`, `median` — сводные значения;
- параметр `axis` указывает ось, которая исчезает после вычисления;
- `argmax` возвращает индекс максимума, а не само максимальное значение;
- `np.matmul(left, right)` и оператор `left @ right` выполняют матричное умножение.

Для массива формы `(rows, columns)` вызов `mean(axis=0)` даёт значение для каждого столбца, а `mean(axis=1)` — для каждой строки.


In [ ]:
import numpy as np

data = np.array([
    [10.0, 20.0, 30.0],
    [12.0, 18.0, 33.0],
    [14.0, 22.0, 27.0],
])

print("greater than 22:", data[data > 22])
print("min/max:", data.min(), data.max())
print("column mean:", data.mean(axis=0))
print("row std:", data.std(axis=1))
print("median:", np.median(data))
print("argmax by row:", data.argmax(axis=1))

weights = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])
print("matmul shape:", np.matmul(data, weights).shape)
print("same with @:", np.array_equal(data @ weights, np.matmul(data, weights)))


#### ❓ **Вопрос**

`data.argmax(axis=1)` вернул `[2, 2, 2]`. Означает ли это, что максимумы в строках равны двум? Какой формы будет результат `data @ weights` для форм `(3, 3)` и `(3, 2)`?

<details>
<summary>Ответ</summary>

Нет, числа `2` — индексы столбцов, где находятся максимумы. Матричное произведение имеет форму `(3, 2)`: внешние размеры остаются, внутренние размеры `3` должны совпасть.

</details>


## 5. NPY и NPZ

NPY хранит один массив вместе с его `shape` и `dtype`. `np.save(path, array)` записывает массив, `np.load(path)` восстанавливает его.

У `np.load` параметр `allow_pickle` по умолчанию равен `False`. Это запрещает автоматическую загрузку Python-объектов из NPY/NPZ и защищает от той же категории риска, что у pickle. Для числовых массивов оставляют `False`.

`mmap_mode="r"` отображает NPY в память и подгружает данные при обращении к срезам. Это полезно, если файл больше доступной RAM, а нужны отдельные строки. Режим `"r"` не разрешает изменять файл.

NPZ хранит несколько именованных массивов. `np.savez` только упаковывает их, а `np.savez_compressed` дополнительно сжимает. Сжатие уменьшает размер не для любых данных и требует времени процессора.


In [ ]:
import numpy as np

features = np.arange(300, dtype=np.float32).reshape(100, 3)
labels = np.arange(100, dtype=np.int16) % 2

features_path = WORK / "features.npy"
dataset_path = WORK / "dataset.npz"

np.save(features_path, features)
np.savez_compressed(dataset_path, features=features, labels=labels)

restored = np.load(features_path, allow_pickle=False)
mapped = np.load(features_path, mmap_mode="r", allow_pickle=False)
print("same:", np.array_equal(features, restored))
print("mapped slice:", mapped[40:43])

with np.load(dataset_path, allow_pickle=False) as archive:
    print("names:", archive.files)
    print(archive["features"].shape, archive["labels"].dtype)


#### ❓ **Вопрос**

Файл NPY занимает 20 ГБ, но программе нужны строки с 1000-й по 1999-ю. Как открыть его без загрузки всех 20 ГБ? Почему совет «поставить `allow_pickle=True`, если файл не читается» опасен?

<details>
<summary>Ответ</summary>

Используют `np.load(path, mmap_mode="r", allow_pickle=False)` и затем берут нужный срез. `allow_pickle=True` разрешает восстановление Python-объектов с выполнением инструкций pickle; неизвестному файлу нельзя давать такое разрешение.

</details>


## 6. HDF5: большое дерево данных в одном файле

HDF5 — один файл с внутренним деревом:

- **group** похожа на каталог и объединяет связанные данные;
- **dataset** похож на массив NumPy, но остаётся на диске;
- **attribute** хранит небольшое описание: единицы, версию, источник.

Например, путь `experiment/values` означает dataset `values` внутри группы `experiment`. У dataset есть `shape`, `dtype` и срезы. `dataset[100:200]` читает выбранные строки, а `dataset[:]` загружает dataset целиком.

Режим `"w"` создаёт файл заново, `"r"` открывает для чтения, `"a"` сохраняет существующее содержимое и разрешает добавление. Блок `with` закрывает файл даже при ошибке.

HDF5 особенно удобен, когда в одном наборе есть несколько больших массивов и метаданные. Это формат файла, не Hadoop HDFS. Подробности про chunks, компрессию и чтение блоками — в «Дополнительно».


In [ ]:
import h5py
import numpy as np

path = WORK / "measurements.h5"
values = np.arange(60, dtype=np.float32).reshape(20, 3)

with h5py.File(path, "w") as file:
    experiment = file.create_group("experiment")
    dataset = experiment.create_dataset(
        "values",
        data=values,
        chunks=(5, 3),
        compression="gzip",
    )
    dataset.attrs["units"] = "mV"
    file.attrs["description"] = "Учебные измерения"

with h5py.File(path, "r") as file:
    dataset = file["experiment/values"]
    print("path:", dataset.name)
    print("shape/dtype:", dataset.shape, dataset.dtype)
    print("units:", dataset.attrs["units"])
    print("three rows:", dataset[10:13])


#### ❓ **Вопрос**

Dataset содержит миллион строк. Чем отличаются `dataset[500:600]` и `dataset[:][500:600]`, хотя результат имеет одинаковые значения?

<details>
<summary>Ответ</summary>

Первый вариант просит HDF5 прочитать только 100 строк. Во втором сначала `dataset[:]` загружает в RAM весь миллион строк и лишь затем NumPy берёт срез.

</details>


## 7. TAR и компрессия

TAR собирает несколько файлов и каталогов в один поток, но сам по себе не обязан сжимать данные:

- `"w"` создаёт обычный `.tar`;
- `"w:gz"` создаёт `.tar.gz` с gzip;
- `"w:xz"` создаёт `.tar.xz` с более медленной, но часто более сильной компрессией;
- `"r:*"` при чтении определяет вариант автоматически.

`archive.add(source, arcname="bundle")` задаёт внутреннее имя вместо абсолютного пути компьютера автора. `archive.extractfile("bundle/meta.json")` открывает один обычный файл внутри TAR без распаковки всего архива на диск; такой объект можно передать в `json.load`. Чтение NPY из TAR через `io.BytesIO` вынесено в «Дополнительно».

Обычный архив сохраняет изменяемые метаданные файлов: время изменения, права и владельца. Заголовок gzip тоже содержит время создания. Поэтому два архива с одинаковым содержимым могут отличаться байтами. Для детерминированного `.tar.gz` нужно:

1. добавлять файлы в отсортированном порядке;
2. установить одинаковые `mtime`, `uid`, `gid`, `uname`, `gname` и `mode` у каждого участника TAR;
3. установить `mtime=0` и пустое имя в заголовке gzip.

Тогда одинаковые входные байты дают одинаковый архив и одинаковую контрольную сумму.

Компрессия хорошо работает на повторяющемся тексте и массивах с закономерностями. Уже сжатые JPEG, MP4 или NPZ могут почти не уменьшиться, а маленький архив иногда становится больше из-за служебных данных. Проверка чужих архивов перед извлечением вынесена в «Дополнительно».


In [ ]:
import gzip
import json
import tarfile

source = WORK / "bundle"
source.mkdir(exist_ok=True)
(source / "readme.txt").write_text("binary data\n" * 100, encoding="utf-8")
(source / "meta.json").write_text('{"count": 100}\n', encoding="utf-8")

gzip_path = WORK / "bundle.tar.gz"

with gzip_path.open("wb") as output:
    with gzip.GzipFile(filename="", mode="wb", fileobj=output, mtime=0) as compressed:
        with tarfile.open(fileobj=compressed, mode="w") as archive:
            for path in sorted(source.rglob("*")):
                if not path.is_file():
                    continue

                name = f"bundle/{path.relative_to(source).as_posix()}"
                info = archive.gettarinfo(path, arcname=name)
                info.mtime = 0
                info.uid = info.gid = 0
                info.uname = info.gname = ""
                info.mode = 0o644

                with path.open("rb") as file:
                    archive.addfile(info, file)

print("tar.gz:", gzip_path.stat().st_size)

with tarfile.open(gzip_path, "r:*") as archive:
    print([member.name for member in archive.getmembers()])
    with archive.extractfile("bundle/meta.json") as file:
        metadata = json.load(file)

print(metadata)


#### ❓ **Вопрос**

Почему два обычных `.tar.gz` с одинаковыми файлами могут иметь разные контрольные суммы? Что фиксирует детерминированный вариант?

<details>
<summary>Ответ</summary>

В заголовки TAR и gzip попадают изменяемые метаданные, прежде всего время. Детерминированный вариант сортирует участников и фиксирует время, владельца, группу, права и данные заголовка gzip. При одинаковом содержимом архивы после этого совпадают побайтно.

</details>


## 8. Что выбирать

| Формат | Что хранит | Чтение части данных | Встроенная компрессия | Типичный выбор |
|---|---|---:|---:|---|
| NPY | один NumPy-массив с формой и типом | да, через `mmap_mode` | нет | промежуточный числовой массив |
| NPZ | несколько именованных массивов | неудобно | по выбору | небольшой набор массивов |
| HDF5 | дерево групп, datasets и атрибутов | да | по dataset | большой структурированный набор |
| pickle | почти любой объект Python | нет | нет | только доверенное внутреннее состояние |
| TAR | несколько готовых файлов | по участникам | через gzip/xz | доставка каталога или результата |


#### ❓ **Вопрос**

Нужно передать коллеге большой HDF5, небольшой JSON с отчётом и README, сохранив имена файлов. Какой формат отвечает за сами большие данные, а какой — за доставку комплекта?

<details>
<summary>Ответ</summary>

HDF5 остаётся форматом большого набора данных. HDF5, JSON и README можно собрать в TAR или TAR.GZ для передачи как одного файла.

</details>


## Дополнительно


### Изолированный генератор случайных чисел

В основной части использован `np.random.seed(seed)`. Он сбрасывает состояние общего генератора и влияет на последующие вызовы `np.random.*` во всей программе. В библиотечном и большом модульном коде удобнее создать отдельный генератор:

```python
rng = np.random.default_rng(42)
sample = rng.normal(size=5)
```

Такой генератор хранит собственное состояние и не изменяет случайные числа в других частях программы.


### `io.BytesIO`: бинарный файл в памяти

`archive.extractfile(...)` возвращает поток участника TAR. Его достаточно для последовательного чтения JSON, но некоторые читатели бинарных форматов ожидают файловый объект с возможностью свободно перемещаться по данным. `io.BytesIO` хранит прочитанные байты в памяти и предоставляет такой интерфейс:

```python
import io
import tarfile
import numpy as np

with tarfile.open("dataset.tar.gz", "r:*") as archive:
    with archive.extractfile("dataset/features.npy") as file:
        buffer = io.BytesIO(file.read())
        features = np.load(buffer, allow_pickle=False)
```

Так загружается только выбранный участник, а не весь TAR. Но сам выбранный NPY сначала целиком попадает в RAM. Для очень большого массива лучше хранить NPY отдельно и использовать `mmap_mode` либо читать HDF5 по срезам.


### Поиск, выбор и сортировка

Частые функции, которые полезно узнавать в рабочем коде:

- `np.where(condition, yes, no)` выбирает значение для каждого элемента;
- `np.where(condition)` и `np.nonzero(condition)` возвращают координаты истинных элементов;
- `np.sort(array, axis=...)` возвращает отсортированную копию;
- `np.argsort(array)` возвращает индексы, задающие порядок;
- `np.unravel_index(flat_index, shape)` переводит один индекс в координаты многомерного массива;
- `np.ptp(array)` вычисляет размах `max - min`;
- `np.unique(array, return_counts=True)` находит уникальные значения и частоты;
- `np.clip(array, low, high)` ограничивает значения диапазоном;
- `np.percentile(array, [25, 50, 75])` считает квантили.

`array.nonzero()` — метод массива; имя функции NumPy пишется слитно: `np.nonzero`, не `non_zero`.


### HDF5: chunks, компрессия и рост dataset

HDF5 хранит chunked dataset блоками. Форма chunk должна соответствовать типичному чтению: если программа постоянно берёт сотни соседних строк, chunk на несколько сотен строк обычно разумнее одной огромной плитки.

```python
dataset = file.create_dataset(
    "values",
    data=values,
    chunks=(1000, values.shape[1]),
    compression="gzip",
    compression_opts=4,
    shuffle=True,
)
```

`gzip` переносим и обычно хорошо сжимает; `lzf` в h5py работает быстрее, но часто сжимает слабее. Компрессия применяется к chunks, поэтому чтение одного элемента всё равно может потребовать чтения и распаковки целого chunk.

Dataset можно создать растущим через `maxshape=(None, columns)` и увеличить методом `resize`. Структуру незнакомого файла смотрят через `list(file.keys())`, `dataset.name`, `shape`, `dtype`, `attrs` или обход `file.visititems(...)`. Для больших данных не вызывают `dataset[:]` без необходимости.


### Пропуски и нормализация

`NaN` обозначает отсутствующее числовое значение. `np.nanmean(data, axis=0)` считает среднее без пропусков. Координаты пропусков можно получить через `np.where(np.isnan(data))`.

Z-нормализация имеет вид `(x - mean) / std`. У постоянного столбца `std == 0`; делить на него нельзя. Один из вариантов — `safe_std = np.where(std == 0, 1, std)`. Исходник сохраняют через `filled = data.copy()`, а результат проверяют вызовом `np.isfinite(normalized).all()`.


### Проверяемый конвейер

Сначала проверяют `shape`, `ndim` и числовой `dtype` исходного массива. Затем создают результат, не меняя исходный файл, сохраняют его и параметры преобразования, повторно открывают созданный файл и проверяют форму, тип и значения.

Средние, стандартные отклонения и другие параметры сохраняют рядом с результатом: они нужны для воспроизводимости и обработки новых данных по тем же правилам.


### Миграция pickle

Доверенный pickle можно загрузить один раз и перевести в более узкие форматы. NumPy-массивы сохраняют через `np.save` или HDF5, JSON-совместимые метаданные — через `json.dump`. После переноса новые файлы читают обратно и сравнивают с исходным объектом. Недоверенный pickle нельзя делать безопасным «предварительной конвертацией»: конвертер всё равно сначала выполнит `pickle.load`.


### Безопасное извлечение TAR

До первого извлечения проверяют **всех** участников. `PurePosixPath(member.name)` помогает обнаружить абсолютный путь и компонент `..`. `member.issym()` и `member.islnk()` находят ссылки, `member.isfile()` — обычный файл.

Отклоняют абсолютные пути, `..`, ссылки и превышение общего лимита распакованного размера. Если один участник опасен, не извлекают ничего. `archive.extract(..., filter="data")` добавляет защиту, но не заменяет собственную проверку политики архива.


### ZIP

ZIP одновременно упаковывает и обычно сжимает каждый файл отдельно. Поэтому из ZIP удобно получить один участник без последовательного чтения всего архива. TAR сначала объединяет файлы в поток, а gzip/xz при необходимости сжимает поток целиком.

```python
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

report_path = Path("report.json")
report_path.write_text('{"status": "ok"}\n', encoding="utf-8")

with ZipFile("bundle.zip", "w", compression=ZIP_DEFLATED) as archive:
    archive.write(report_path, arcname="bundle/report.json")

with ZipFile("bundle.zip") as archive:
    print(archive.namelist())
    with archive.open("bundle/report.json") as file:
        content = file.read()
```

Для ZIP действуют те же правила доверия к путям участников: перед массовым извлечением проверяют абсолютные пути и `..`.
